# Data Preprocessing
# Mushroom Observations (iNaturalist)

Builds the cleaned, labeled dataset used for exploration ([`02_exploratory_analysis.ipynb`](02_exploratory_analysis.ipynb)) and model training ([`03_model_training_vgg19.ipynb`](03_model_training_vgg19.ipynb)).

**Source:** raw export from [iNaturalist](https://www.inaturalist.org/observations/export) (`observations-692715.csv`) user-submitted wildlife observations with coordinates, dates, species names, and photo URLs, in this case it's restricted to fungi.

**Pipeline:**
1. Load the raw export
2. Match each observation to a locally downloaded image
3. Keep only Fungi observations
4. Select the columns needed downstream
5. Map each species to an edibility label and a short description
6. Drop species with too few observations to train on
7. Export `mushroom.csv`

In [5]:
import pandas as pd 

df = pd.read_csv("../Data/observations-692715.csv")

print(df.sample(5))
print(df.shape)
print(df.describe(include='all').T)

              id                                  uuid  \
30386  276505870  fb8d7456-1b67-4c10-a8dc-fa777947960f   
24945  218804195  1cd06a4f-8039-445f-809a-243befa60fbc   
1732    11376323  075e0537-730a-4382-ad71-06fe746064ee   
31173  286975597  7cb8b409-019d-448f-9886-34adeca67b9a   
16721  144373657  694d9c85-725e-4891-a99a-542e54d2ba2f   

                               observed_on_string observed_on  \
30386                          2025/04/28 9:25 AM  2025-04-28   
24945                            2023/09/10 14:00  2023-09-10   
1732   Tue Apr 24 2018 11:17:42 GMT+1200 (GMT+12)  2018-04-24   
31173                         2025-06-05 16:57:54  2025-06-05   
16721                            2022/11/26 16:47  2022-11-26   

                time_observed_at      time_zone  user_id          user_login  \
30386  2025-04-28 07:55:00 +0200         Tehran   613173         shahrzadasa   
24945  2023-09-10 08:00:00 +0200         Taipei  8037190          shulinchen   
1732   2018-04-24 01

## 2 · Match observations to downloaded images

Images were downloaded separately (see the commented-out script below, kept for reference) into `Data/images/`, named `img_{row_index}.{ext}`. Here we look up which of those files actually exist on disk and attach the filename to each row; observations with no matching image are dropped.

In [6]:
#DOWNLOAD IMAGES
#
#import requests
#import os
#
#url_column = "image_url"
#
#os.makedirs("images", exist_ok=True)
#
#image_names = []
#
#for i, url in enumerate(df[url_column]):
#    filename = f"img_{i}.jpg"
#    filepath = os.path.join("images", filename)
#
#    try:
#        response = requests.get(url, timeout=10)
#        if response.status_code == 200:
#            with open(filepath, "wb") as f:
#                f.write(response.content)
#            image_names.append(filename)
#        else:
#            image_names.append(None)
#    except Exception as e:
#        print(f"Failed {url}: {e}")
#        image_names.append(None)
#
#df["image_filename"] = image_names
#

In [9]:
import os

folder_path = "../Data/images/"
existing_files = set(os.listdir(folder_path))

image_list = []

for i in range(len(df)):
    found = False
    
    for ext in [".png", ".jpg", ".jpeg"]:
        filename = f"img_{i}{ext}"
        if filename in existing_files:
            image_list.append(filename)
            found = True
            break
    
    if not found:
        image_list.append("no image")

df["image_filename"] = image_list
df = df[df["image_filename"] != "no image"]

## 3 · Keep Fungi observations only

The raw export isn't fungi-specific (`iconic_taxon_name` covers all iNaturalist taxa), restrict to `"Fungi"`.

In [10]:
df = df[df["iconic_taxon_name"] == "Fungi"]

## 4 · Select the columns needed downstream

Drop everything not needed for exploration or training (raw iNaturalist metadata), keeping identifiers, location, date, species names, image filename, and description.

In [11]:
col_to_keep = ["id", "latitude", "longitude", "observed_on", "place_guess",
               "place_country_name", "species_guess", "scientific_name", "common_name", "image_filename", "description" ]

df = df[col_to_keep]

df.describe(include='all')

,id,latitude,longitude,observed_on,place_guess,place_country_name,species_guess,scientific_name,common_name,image_filename,description
count,4.146000e+03,4135.000000,4135.000000,4146,4132,4135,4140,4146,4145,4146,1059
unique,NaN,NaN,NaN,1463,3094,67,126,29,28,4146,1029
top,NaN,NaN,NaN,2019-10-12,"Santa Clara County, US-CA, US",United States,Yellow Stainer,Agaricus xanthodermus,Agaric jaunissant,img_4710.jpg,"San Diego County, California, US"
freq,NaN,NaN,NaN,23,58,2212,1012,1112,1112,1,8
mean,1.752320e+07,25.280292,-36.886317,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1.071494e+07,32.111825,110.333510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,5.410000e+02,-47.284153,-177.901783,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,8.361318e+06,32.725400,-122.193273,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,1.720070e+07,37.802022,-99.698276,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2.731225e+07,44.370638,12.328374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5 · Map species: edibility & description

`scientific_name` alone isn't useful to an end user, build a lookup (`species_info`) with a hand-curated edibility label (`edible` / `poisonous` / `inedible` / `unknown`) and a plain-language description per species, covering the 47 species present in the data (including non-fungi taxa like corals and beetles that share genus-level names in the export). Species missing from the lookup default to `"unknown"`.

In [12]:
species_info = {
    "Agaricus augustus": {"edibility": "edible", "new_description": "Large edible mushroom with strong almond-like smell, found in grasslands and forests."},
    "Agaricus arvensis": {"edibility": "edible", "new_description": "Edible 'horse mushroom,' grows in fields and meadows, white cap, pleasant taste."},
    "Auricularia cornea": {"edibility": "edible", "new_description": "Edible jelly fungus, dark brown, often called 'wood ear,' grows on dead wood."},
    "Agaricus xanthodermus": {"edibility": "poisonous", "new_description": "Poisonous yellow-staining mushroom, often mistaken for edible Agaricus."},
    "Lobactis scutaria": {"edibility": "unknown", "new_description": "Coral-like marine organism, not a mushroom; inedible."},
    "Agaricus sylvaticus": {"edibility": "edible", "new_description": "Edible woodland mushroom, brown cap, grows in woods and leaf litter."},
    "Agaricus campestris": {"edibility": "edible", "new_description": "Edible field mushroom, white cap, common in grasslands, similar to store mushrooms."},
    "Marasmius oreades": {"edibility": "edible", "new_description": "Edible 'fairy ring mushroom,' small, grows in rings in grass, mild flavor."},
    "Agaricus": {"edibility": "unknown", "new_description": "Genus of mushrooms; includes edible and poisonous species, needs species-level ID."},
    "Melochia tomentosa": {"edibility": "inedible", "new_description": "Tropical flowering plant; not a mushroom; inedible."},
    "Agaricus moelleri": {"edibility": "poisonous", "new_description": "Poisonous 'inhabited mushroom,' causes gastrointestinal upset."},
    "Agaricus bernardii": {"edibility": "edible", "new_description": "Edible, often salty-tasting mushroom, grows in coastal or disturbed soils."},
    "Fungia fungites": {"edibility": "inedible", "new_description": "Marine coral species, not a mushroom; inedible."},
    "Cynomorium coccineum": {"edibility": "inedible", "new_description": "Parasitic flowering plant, red stems, sometimes used in traditional medicine; inedible."},
    "Sarcophyton": {"edibility": "inedible", "new_description": "Soft coral genus, not a mushroom; inedible."},
    "Auricularia nigricans": {"edibility": "edible", "new_description": "Edible jelly fungus, similar to wood ear, dark brown, grows on logs."},
    "Agaricus sylvicola": {"edibility": "edible", "new_description": "Edible woodland Agaricus, found under trees, similar to A. campestris."},
    "Agaricus bisporus": {"edibility": "edible", "new_description": "Common edible mushroom (button/cremini/portobello), widely cultivated."},
    "Agaricus bitorquis": {"edibility": "edible", "new_description": "Edible 'spring agaricus,' grows in grass, cap brown, edible and tasty."},
    "Ctenactis albitentaculata": {"edibility": "inedible", "new_description": "Stony coral species, not a mushroom; inedible."},
    "Ctenactis crassa": {"edibility": "inedible", "new_description": "Stony coral species, not a mushroom; inedible."},
    "Agaricus crocodilinus": {"edibility": "unknown", "new_description": "Rare Agaricus species, edibility unclear; unknown."},
    "Tetratoma fungorum": {"edibility": "inedible", "new_description": "Fungus beetle species, not a mushroom; inedible."},
    "Agaricus benesii": {"edibility": "edible", "new_description": "Woodland mushroom, edible but uncommon."},
    "Agaricus essettei": {"edibility": "unknown", "new_description": "Edibility unknown; little documented information."},
    "Ctenactis echinata": {"edibility": "inedible", "new_description": "Stony coral species, not a mushroom; inedible."},
    "Agaricus phaeolepidotus": {"edibility": "unknown", "new_description": "Woodland Agaricus, edibility unknown; rare."},
    "Agaricus bohusii": {"edibility": "edible", "new_description": "Edible Agaricus, small to medium-sized, found in forests."},
    "Agaricus comtulus": {"edibility": "edible", "new_description": "Edible woodland mushroom, grows in Europe; small brown cap."},
    "Polyphyllia talpina": {"edibility": "inedible", "new_description": "Coral species, not a mushroom; inedible."},
    "Agaricus langei": {"edibility": "edible", "new_description": "Edible Agaricus, found in leaf litter; medium size."},
    "Agaricus brunneolus": {"edibility": "unknown", "new_description": "Small brown Agaricus, edibility uncertain; not well-documented."},
    "Agaricus subfloccosus": {"edibility": "unknown", "new_description": "Rare Agaricus, edibility uncertain."},
    "Minores": {"edibility": "unknown", "new_description": "Likely a shorthand or subgenus; insufficient info; unknown."},
    "Agaricus dulcidulus": {"edibility": "edible", "new_description": "Edible, small forest mushroom; mild taste."},
    "Agaricus litoralis": {"edibility": "edible", "new_description": "Edible coastal Agaricus, found in sandy soils."},
    "Agaricus cupreobrunneus": {"edibility": "edible", "new_description": "Edible woodland mushroom, medium brown cap."},
    "Agaricus devoniensis": {"edibility": "unknown", "new_description": "Rare Agaricus species; edibility unknown."},
    "Agaricus bresadolanus": {"edibility": "edible", "new_description": "Edible woodland mushroom; small, brown cap."},
    "Agaricus lanipes": {"edibility": "edible", "new_description": "Edible Agaricus, grows in leaf litter; medium size."},
    "Pleuractis paumotensis": {"edibility": "inedible", "new_description": "Coral species, not a mushroom; inedible."},
    "Agaricus subperonatus": {"edibility": "unknown", "new_description": "Edibility unknown; rare forest Agaricus."},
    "Agaricus depauperatus": {"edibility": "unknown", "new_description": "Rare woodland Agaricus; edibility unknown."},
    "Agaricus altipes": {"edibility": "edible", "new_description": "Edible Agaricus; small to medium size, found in woods."},
    "Saprolegnia parasitica": {"edibility": "inedible", "new_description": "Water mold, not a mushroom; inedible."},
    "Agaricus impudicus": {"edibility": "unknown", "new_description": "Edibility uncertain; uncommon Agaricus."},
    "Danafungia horrida": {"edibility": "inedible", "new_description": "Stony coral species, not a mushroom; inedible."}
}

df["edibility"] = df["scientific_name"].map(lambda x: species_info.get(x, {}).get("edibility", "unknown"))
df["new_description"] = df["scientific_name"].map(lambda x: species_info.get(x, {}).get("new_description", "No description available"))

In [13]:
df.describe(include='all')

,id,latitude,longitude,observed_on,place_guess,place_country_name,species_guess,scientific_name,common_name,image_filename,description,edibility,new_description
count,4.146000e+03,4135.000000,4135.000000,4146,4132,4135,4140,4146,4145,4146,1059,4146,4146
unique,NaN,NaN,NaN,1463,3094,67,126,29,28,4146,1029,3,29
top,NaN,NaN,NaN,2019-10-12,"Santa Clara County, US-CA, US",United States,Yellow Stainer,Agaricus xanthodermus,Agaric jaunissant,img_4710.jpg,"San Diego County, California, US",edible,"Poisonous yellow-staining mushroom, often mist..."
freq,NaN,NaN,NaN,23,58,2212,1012,1112,1112,1,8,2928,1112
mean,1.752320e+07,25.280292,-36.886317,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1.071494e+07,32.111825,110.333510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,5.410000e+02,-47.284153,-177.901783,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,8.361318e+06,32.725400,-122.193273,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,1.720070e+07,37.802022,-99.698276,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2.731225e+07,44.370638,12.328374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6 · Drop species with too few observations

Some of the 47 species have only a handful of observations, not enough to train a classifier on. Keep only species with at least 50 observations, which narrows the dataset to the species actually usable for the model in [`03_model_training_vgg19.ipynb`](03_model_training_vgg19.ipynb).

In [14]:
df = df[df['scientific_name'].map(df['scientific_name'].value_counts()) >= 50]

In [15]:
df["scientific_name"].value_counts()

scientific_name
Agaricus xanthodermus    1112
Auricularia cornea        698
Marasmius oreades         676
Agaricus campestris       565
Agaricus augustus         536
Agaricus bitorquis        121
Agaricus bernardii        116
Agaricus arvensis          95
Agaricus                   52
Name: count, dtype: int64

## 7 · Export the cleaned dataset

Save the result as `mushroom.csv`, the input to [`02_exploratory_analysis.ipynb`](02_exploratory_analysis.ipynb) and (after further curation) model training.

In [18]:
df.to_csv("../Data/mushroom.csv")